# Rapport final : Python pour la data-science 2A S1

Manon Bourguignat, Leila Héry, Franck Tran  
Chargé de TD : Lino Galiana

On pensera à lire le fichier README.md du repo qui contient notamment les informations sur les dépendances et les données.

## I. Contexte

### 1. Présentation du sujet

La pandémie de Covid-19 a ravivé et intensifié une méfiance marquée envers les discours officiels au sein des démocraties européennes. Loin de constituer un simple manque d'information individuel, l'adhésion aux thèses complotistes reflète souvent une rupture démocratique et épistémique plus profonde vis-à-vis des institutions établies.

À partir des données de l'European Social Survey (ESS Round 10, 2020), notre étude vise à dépasser les seuls déterminants socio-économiques traditionnels (diplôme, revenu). Nous distinguons notamment la défiance envers le système politique de celle dirigée contre le consensus scientifique, tout en mesurant l'effet de la radicalité idéologique (éloignement du centre sur l'axe gauche-droite) et de l'ancrage territorial.

Notre travail s'articule ainsi autour de la problématique suivante :

> **« Dans quelle mesure l’adhésion aux théories du complot en Europe traduit-elle une défiance institutionnelle et scientifique plutôt qu’un clivage socio-économique traditionnel ? »**


### 2. Présentation de la base de donnée utilisée

L'European Social Service (ESS) est une enquête menée régulièrement dans plusieurs pays européens pour étudier les opinions, les valeurs et les comportements des populations. Chaque round (correspondant à une enquête réalisée une année donnée) comprend des variables communes à l'ensemble des rounds, ainsi que des variables propres aux thèmes étudiés.

Dans ce projet, on utilisera les données de l’ESS Round 10 (noté égament ESS10), édition 3.3, collectées en 2020 dont les thèmes sont la démocratie et les contacts sociaux numériques.

Tous les détails (références, licenses, accès) concernant les données se situent dans le fichier README.md.

## II. Préparation des données

Cette partie consiste en la récupération, l'exploration et le nettoyage des données. Cette partie se conclut sur la création d'un dataframe propre nommé `df_final` qui sera utilisé dans la partie III.

De nombreuses fonctions et variables utilisées sont regroupées dans des scripts python, dans le dossier "scripts" qu'on importe ici :

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# On importe des fonctions définies dans preparation_data.py
from scripts.preparation_data import recuperer_donnees, select_colonne, suppr_colonne, renommer_colonne 

# On importe aussi les variables
from scripts.preparation_data import dico_doi, colonnes_a_garder, nouveaux_noms, complot_cols, inst_cols

### 1. Récupération des données

Le choix a été de récupérer les données de l'enquête de 2020 avec l'API dédiée pour deux raisons :
- Faciliter la reproductibilité : l'API est plus pratique que de chercher à récupérer les données sur le site
- L'infrastructure de recherche qui gère l'ESS recommande dans ses conditions d'utilisation (https://europeansocialsurvey.org/contact/disclaimer) de ne pas mettre les jeux de données de l’ESS à disposition sur des sites web externes.

Lorsqu’on exécute la fonction `recuperer_donnes` définie dans preparation_data.py, **il faut renseigner un "ESS User ID"**.

<details style="border: 1px solid #888; border-radius: 6px; padding: 8px;">
<summary style="font-weight: bold; cursor: pointer;">Où trouver mon ESS User ID ?</summary>

 Elle est disponible sur la page de l’API de l’ESS (https://ess.sikt.no/en/api) une fois connecté à un compte.
 
 On pourra également utiliser l'ESS User ID de Franck TRAN qu'il vous aura transmis par une autre voie. 
 
 Cet identifiant ne sert pas à s’authentifier, mais à suivre les statistiques d’utilisation des données. 

In [161]:
df_ess_2020 = recuperer_donnees(2020)
display(df_ess_2020.head(5))

,name,essround,edition,proddate,idno,cntry,dweight,pspwght,pweight,anweight,...,vinwe,inwde,jinws,jinwe,inwtm,mode,domain,prob,stratum,psu
0,ESS10e03_3,10,3.3,02.12.2025,10038,BE,0.882220,0.972276,0.718075,0.698167,...,2022-09-01 17:47:00,2022-09-01 17:47:00,2022-09-01 17:47:00,2022-09-01 17:47:00,36.0,1.0,1,0.000397,188,2596
1,ESS10e03_3,10,3.3,02.12.2025,10053,BE,1.047642,0.888635,0.718075,0.638107,...,2022-04-08 11:07:00,2022-04-08 11:10:00,2022-04-08 11:07:00,2022-04-08 11:10:00,54.0,2.0,2,0.000334,194,2206
2,ESS10e03_3,10,3.3,02.12.2025,10055,BE,1.087741,0.722811,0.718075,0.519033,...,2022-05-20 11:08:00,2022-05-20 11:10:00,2022-05-20 11:08:00,2022-05-20 11:10:00,77.0,1.0,2,0.000322,198,2114
3,ESS10e03_3,10,3.3,02.12.2025,10062,BE,0.909910,1.005565,0.718075,0.722072,...,2022-05-22 13:58:00,2022-05-22 13:59:00,2022-05-22 13:58:00,2022-05-22 13:59:00,55.0,1.0,1,0.000385,150,2645
4,ESS10e03_3,10,3.3,02.12.2025,10064,BE,0.918949,0.638705,0.718075,0.458639,...,2022-05-18 11:44:00,2022-05-18 11:45:00,2022-05-18 11:44:00,2022-05-18 11:45:00,55.0,1.0,1,0.000381,149,2313


On observe qu'il y a +37000 échantillons et +621 variables avec des noms pas très explicites comme `idno` pour désigner le numéro d’identification du répondant.

On conserve uniquement les colonnes susceptibles de contribuer à la réponse à la problématique avec `select_colonne` et la variable `colonnes_a_garder` (= liste contenant toutes les noms des colonnes à garder).

De même, `rename_colonne` et `nouveaux_noms` nous permettent de renommer :

In [162]:
df_ess_select = select_colonne(df_ess_2020, colonnes_a_garder)
df_ess_rename = renommer_colonne(df_ess_select, colonnes_a_garder, nouveaux_noms)
display(df_ess_rename.head(5))

,id,pays,poids_analyse,poids_design,poids_poststrat,complot_groupes_secrets,complot_science_politique,dissimulation_covid,mesures_sanitaires_controle,mesures_sanitaires_equilibre,...,activite_principale,chomage,milieu_residence,appartenance_religieuse,degre_religiosite,frequence_culte,frequence_priere,genre,age,annee_naissance
0,10038,BE,0.698167,0.882220,0.972276,4.0,3.0,4.0,7.0,2.0,...,NaN,0,4.0,1.0,7.0,3.0,1.0,2,16.0,2006.0
1,10053,BE,0.638107,1.047642,0.888635,3.0,3.0,3.0,4.0,6.0,...,NaN,1,4.0,2.0,4.0,7.0,7.0,2,24.0,1998.0
2,10055,BE,0.519033,1.087741,0.722811,4.0,5.0,5.0,7.0,5.0,...,NaN,0,4.0,2.0,0.0,7.0,7.0,1,58.0,1964.0
3,10062,BE,0.722072,0.909910,1.005565,NaN,NaN,5.0,9.0,5.0,...,NaN,0,1.0,1.0,5.0,7.0,6.0,1,35.0,1987.0
4,10064,BE,0.458639,0.918949,0.638705,5.0,5.0,5.0,9.0,5.0,...,NaN,0,2.0,2.0,1.0,7.0,7.0,1,61.0,1961.0


### 2. Valeurs manquantes

D'après la documentation, il y a de nombreuses formes possibles de non réponses (comme un `Refus`, `Ne sait pas` ou `Pas de réponse` qui sont codés par des valeurs particulières). Cependant, grâce à l'API, les données qu'on a récupéré ci-dessus sont déjà recodés en `np.nan`.

Regardons alors les variables ayant +5% de valeurs manquantes.

In [172]:
def show_df_val_manq(df):
    df_val_manquant = pd.DataFrame({
        "Nbr valeurs manquantes": df.isnull().sum(),
        "% valeurs manquantes": df.isnull().mean() * 100
    })

    df_val_manquant = df_val_manquant.sort_values(by="% valeurs manquantes", ascending=False).round(2)

    return df_val_manquant

# Ce dataframe des valeurs manquantes sera réutilisé par la suite
df_val_manq_avant = show_df_val_manq(df_ess_rename)

display(df_val_manq_avant[df_val_manq_avant["% valeurs manquantes"] >= 5])

,Nbr valeurs manquantes,% valeurs manquantes
refus_vaccin,37073,98.57
activite_principale,33940,90.24
degre_proximite_parti,23289,61.92
statut_vaccine,20679,54.98
decile_revenu,8239,21.91
confiance_science,7866,20.91
dissimulation_covid,5762,15.32
complot_groupes_secrets,5632,14.97
axe_gauche_droite,5297,14.08
complot_science_politique,5293,14.07


On supprime les variables `refus_vaccin`, `activite_principale`, `degre_proximite_parti`, `statut_vaccine` qui ont une part trop élevée de valeurs manquantes :

In [173]:
df_ess_suppr = suppr_colonne(df_ess_rename, 
    ["refus_vaccin", "activite_principale", "degre_proximite_parti", "statut_vaccine"])

print(df_ess_suppr.shape)

(37611, 35)


Quant aux autres variables ayant entre 10 % et 20 % de valeurs sont manquantes, on fait le choix de ne pas les supprimer. Lors de l'analyse descriptive, les valeurs manquantes seront exclues du calcul et regroupées dans une catégorie `Non_renseigné`. 

En consultant la documentation méthodologique officielle de l'ESS Round 10 (*voir fichier README.md*), on remarque une absence totale de certaines valeurs dans certains pays, s'expliquant par les contraintes imposées par la crise sanitaire sur les modes de collecte.

<details style="border: 1px solid #888; border-radius: 6px; padding: 8px;">
<summary style="font-weight: bold; cursor: pointer;">Explication détaillée des valeurs manquantes à l'aide de la documentation</summary>

1. **Raccourcissement du questionnaire pour l'auto-passation (Self-Completion) :**  
   Plusieurs pays, dont le Monténégro (ME) et le Royaume-Uni (GB), ont dû abandonner le traditionnel entretien en face-à-face pour basculer en urgence vers un mode d'auto-administration (questionnaire web et papier par courrier). Pour limiter le fardeau cognitif et les taux d'abandon inhérents aux questionnaires sans enquêteur, l'ESS a drastiquement allégé le questionnaire, supprimant certains modules rotatifs secondaires dont les questions spécifiques sur la science (`trstsci`) ou les décisions secrètes.

2. **Divergences nationales d'implémentation du module Covid-19 :**  
   Pour la France (FR), le terrain d'enquête a été décalé et restructuré sous le format face-à-face assisté par ordinateur (CAPI), mais avec des arbitrages temporels stricts sur la durée de passation. L'équipe nationale de coordination a priorisé le tronc commun et le module démocratie, omettant l'extension facultative consacrée à la perception de la science et aux attitudes sanitaires alternatives.

Ces valeurs manquantes ne relèvent donc pas d'une non-réponse individuelle aléatoire (*Missing at Random*), mais d'un choix institutionnel de conception de l'enquête (*Structural Missings* / *Not Available*).

</details>

Pour notre analyse, cela implique de restreindre notre analyse aux pays ayant administré l'intégralité des questions. Regardons ainsi quels sont les pays où il y a beaucoup de valeurs manquantes :

In [174]:
# Soit les variables ayant entre 5% et 21% de valeurs manquantes
colonne_ayant_bcp_val_manq = ["decile_revenu", "confiance_science", "dissimulation_covid",
    "complot_groupes_secrets", "axe_gauche_droite", "complot_science_politique",
    "mesures_sanitaires_equilibre", "mesures_sanitaires_controle"]

df_par_pays = select_colonne(df_ess_suppr, colonne_ayant_bcp_val_manq)
df_par_pays = df_ess_suppr.groupby("pays").agg(lambda col: col.isnull().mean() * 100)

# Pour chaque variable e, regardons quels pays ont +20% de valeurs manquantes
for e in colonne_ayant_bcp_val_manq:
    df_temporaire = df_par_pays.sort_values(by=[e], ascending=False)
    display(df_temporaire.loc[df_temporaire[e] >= 20, [e]]
            .reset_index()
            .style.hide(axis="index")
            .format({e: "{:.2f}"}))


pays,decile_revenu
GR,46.09
IT,38.37
PT,34.28
IE,32.99
HR,30.53
CZ,30.45
SK,29.97
HU,26.93
MK,23.58


pays,confiance_science
EE,100.00
CZ,100.00
ME,100.00
FR,100.00


pays,dissimulation_covid
ME,100.00
FR,100.00
PT,23.01


pays,complot_groupes_secrets
ME,100.00
FR,100.00


pays,axe_gauche_droite
MK,38.42
ME,32.24
LT,26.94
IT,25.80
PT,20.78


pays,complot_science_politique
ME,100.00
FR,100.00


pays,mesures_sanitaires_equilibre
ME,100.00
FR,100.00


pays,mesures_sanitaires_controle
ME,100.00
FR,100.00


Pour `décile_revenu` et `axe_gauche_droite`, les valeurs manquantes se répartissent entre de nombreux pays.

Pour le reste des variables, l'examen détaillé des données révèle que le volume élevé de valeurs manquantes n'est pas homogène : il résulte principalement d'une absence totale de passation de ces variables dans certains pays (notamment 100 % de valeurs manquantes pour `confiance_science` en Estonie, Tchéquie, Monténégro et France). 

Afin de préserver la validité des indicateurs synthétiques qu'on créera et de garantir la comparabilité des résultats, il apparaît donc préférable d'exclure ces pays du champ de notre étude principale, quitte à réduire légèrement la taille de l'échantillon global.

In [175]:
df_ess_suppr_pays = df_ess_suppr[~df_ess_suppr["pays"].isin(["FR", "ME", "EE", "CZ"])]

print(df_ess_suppr_pays.shape, "\n")

(30338, 35) 



Comparons avec les valeurs manquantes au début.

In [176]:
df_val_manq_apres = show_df_val_manq(df_ess_suppr_pays)

# On joint les 2 dataframes des valeurs manquantes
df_val_manq_joint = df_val_manq_avant.merge(
    df_val_manq_apres,
    left_index=True,
    right_index=True,
    how="inner",
    suffixes=(" avant", " apres"),)

# On crée une nouvelle colonne
df_val_manq_joint["Variation en % des valeurs manquantes"] = (
    df_val_manq_joint["% valeurs manquantes apres"]
    - df_val_manq_joint["% valeurs manquantes avant"])

# On affiche ce qui nous intéresse
display(df_val_manq_joint
    .sort_values(by="% valeurs manquantes apres", ascending=False)
    .loc[
        df_val_manq_joint["% valeurs manquantes avant"] >= 5,
        ["% valeurs manquantes avant", "% valeurs manquantes apres", "Variation en % des valeurs manquantes"]
])

,% valeurs manquantes avant,% valeurs manquantes apres,Variation en % des valeurs manquantes
decile_revenu,21.91,23.38,1.47
axe_gauche_droite,14.08,14.00,-0.08
dissimulation_covid,15.32,7.67,-7.65
complot_groupes_secrets,14.97,7.18,-7.79
complot_science_politique,14.07,6.05,-8.02
mesures_sanitaires_equilibre,13.54,5.49,-8.05
confiance_science,20.91,1.95,-18.96
mesures_sanitaires_controle,10.44,1.94,-8.50


On observe ainsi que mise à part le décile de revenu, le taux de valeurs manquantes a bien diminué dans les autres variables.

### 3. Création de variables agrégées


Afin d'opérationnaliser notre problématique sans surcharger l'analyse, nous avons fait le choix de réduire drastiquement le nombre de dimensions en créant des indicateurs synthétiques dans un dataframe final nommé `df_final`. Au vu de la structure du jeu de données, nous restreignons notre analyse à un ensemble final de 10 variables clés : 3 variables expliquées et 7 variables explicatives (mêlant indicateurs construits et variables brutes directement pertinentes).

1. **Variables expliquées (mesure de l'adhésion aux complots) :**
   * `score_complot` *(créée)* : indicateur continu (de 0 à 10) mesurant le niveau moyen d'adhésion aux récits complotistes (moyenne synthétique des questions sur les groupes secrets, la collusion politico-scientifique et le Covid-19).
   * `adhesion_complot_forte` *(créée)* : indicatrice binaire (0 ou 1) identifiant les profils complotistes affirmés (accord $\ge 4/5$ sur au moins deux items), adaptée aux futures régressions logistiques.
   * `dissimulation_covid` *(brute)* : variable ordinale mesurant la perception de rétention d'informations par les autorités lors de la crise sanitaire.

2. **Variables explicatives (déterminants politiques, culturels et structurels) :**
   * `confiance_institutions` *(créée)* : score continu (0 à 10) agrégeant la confiance dans le parlement, le personnel politique, les partis et la justice.
   * `score_socioeco` *(créée)* : indicateur composite continu (borné de 0 à 1, inspiré de l'IDH) combinant le décile de revenu du ménage et le nombre d'années d'études.
   * `radicalite_politique` *(créée)* : distance absolue au centre (5) sur l'axe gauche-droite pour capter l'effet des extrêmes.
   * `zone_rurale` *(créée)* : indicatrice binaire distinguant l'habitat rural/campagne du milieu urbain.
   * `confiance_science` *(brute)* : échelle continue (0 à 10) mesurant la confiance envers le consensus scientifique.
   * `degre_religiosite` *(brute)* : échelle continue (0 à 10) mesurant l'intensité de la foi personnelle.
   * `age` *(brute)* : âge exact du répondant pour contrôler les effets de cohorte.

Ce resserrement méthodologique permet de disposer d'indicateurs continus robustes pour les régressions linéaires tout en évitant les problèmes de multicolinéarité liés à la multiplication de questions similaires.



In [ ]:
df_clean = df_ess_suppr_pays.copy()

# 1) construction des variables expliquées (Y)
    # A) score synthétique continu de complotisme ramené sur une échelle de 0 à 10
df_clean["score_complot"] = (df_clean[complot_cols].mean(axis=1) - 1) * 2.5

    # B) indicatrice d'adhésion forte (binaire : 1 si accord fort sur >= 2 items, 0 sinon)
df_clean["adhesion_complot_forte"] = ((df_clean[complot_cols] >= 4).sum(axis=1) >= 2).astype(int)


# 2) construction des variables explicatives synthétiques (X)
    # C) confiance institutionnelle moyenne (continue, 0 à 10)
df_clean["confiance_institutions"] = df_clean[inst_cols].mean(axis=1)

    # D) score socio-économique composite normalisé (continu de 0 à 1)
rev_norm = (df_clean["decile_revenu"] - 1) / 9.0
edu_norm = df_clean["annees_etudes"] / 25.0
df_clean["score_socioeco"] = (rev_norm + edu_norm) / 2.0

    # E) radicalité politique (distance au centre 5 sur l'axe gauche-droite)
df_clean["radicalite_politique"] = (df_clean["axe_gauche_droite"] - 5).abs()

    # F) zone rurale (binaire : 1 si campagne/village, 0 si ville)
df_clean["zone_rurale"] = df_clean["milieu_residence"].isin([4, 5]).astype(int)


# 3) définition des listes de variables pour les régressions et visualisations

variables_expliquees = [
    "score_complot", 
    "adhesion_complot_forte", 
    "dissimulation_covid"
]

variables_explicatives = [
    "confiance_institutions",
    "confiance_science",
    "radicalite_politique",
    "score_socioeco",
    "degre_religiosite",
    "zone_rurale",
    "age"
]

variables_finales = variables_expliquees + variables_explicatives

df_clean = select_colonne(df_clean, variables_finales)
print(df_clean.shape)
display(df_clean[variables_finales].head())


,score_complot,adhesion_complot_forte,dissimulation_covid,confiance_institutions,confiance_science,radicalite_politique,score_socioeco,degre_religiosite,zone_rurale,age
0,6.666667,1,4.0,6.666667,8.0,2.0,NaN,7.0,1,16.0
1,5.000000,0,3.0,4.500000,4.0,0.0,NaN,4.0,1,24.0
2,9.166667,1,5.0,6.250000,9.0,0.0,0.693333,0.0,1,58.0
3,10.000000,0,5.0,8.500000,8.0,0.0,0.573333,5.0,0,35.0
4,10.000000,1,5.0,6.750000,9.0,1.0,0.657778,1.0,0,61.0


## III. Visualisation 

Cette partie consiste en l'analyse et la visualsation de notre jeu de données final pour répondre à la problématique.

### 1. Statistiques descriptives

In [ ]:
# faire dataframe ou graphiques comme dans la 6 du fichier stats desc qui monte les stats comme moyenne, etc. 
# des différentes variables

Faire des "commentaires dessus"

### 2. Analyse descriptive

Graphiques basique (histogramme, boxplot qui permet situer)

Graphiques détaillant relation entre les variables (genre confiance_science en fonction du decile revenu etc..)

## IV. Conclusion

Ceci est une conclusion